In [8]:
import ee
import pandas as pd

# ==========================================
# 1. LIGANDO A CONEXÃO COM O GOOGLE
# ==========================================
# ee.Authenticate() # Deixe com o '#' se já rodou e autenticou com sucesso
ee.Initialize(project='atmosalert-497701')

# ==========================================
# 2. EXTRAÇÃO DOS DADOS DE FUMAÇA
# ==========================================
# Puxando a coleção do satélite Sentinel-5P para agosto de 2023
s5p_colecao = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_AER_AI')
               .filterDate('2023-08-01', '2023-08-31')
               .select('absorbing_aerosol_index'))

# Definindo o ponto de interesse (Campinas)
ponto_campinas = ee.Geometry.Point([-47.06, -22.90])

def extrair_serie_temporal(imagem):
    dict_valor = imagem.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=ponto_campinas,
        scale=1113.2,
        bestEffort=True
    )
    data = imagem.date().format('YYYY-MM-dd')
    
    return ee.Feature(None, {
        'Data': data,
        'Indice_Fumaca': dict_valor.get('absorbing_aerosol_index')
    })

# Mapeando a função
dados_brutos = s5p_colecao.map(extrair_serie_temporal)

# ==========================================
# 3. CONVERSÃO À PROVA DE FALHAS (NATIVA)
# ==========================================
# Baixa os dados da nuvem do Google em formato de dicionário (JSON)
info = dados_brutos.getInfo()

# Extrai apenas as propriedades (Data e Indice_Fumaca) de cada leitura
lista_dados = [feature['properties'] for feature in info['features']]

# Converte diretamente com o Pandas, sem depender do geemap!
df_fumaca = pd.DataFrame(lista_dados)

# Limpeza e ordenação dos dados
df_fumaca = df_fumaca.dropna()
df_fumaca = df_fumaca.sort_values(by='Data').reset_index(drop=True)

# Salvando em CSV
df_fumaca.to_csv('historico_fumaca_campinas.csv', index=False)

print("✅ Dados extraídos com sucesso! Arquivo CSV gerado.")
display(df_fumaca.head())

✅ Dados extraídos com sucesso! Arquivo CSV gerado.


,Data,Indice_Fumaca
0,2023-08-01,-0.126956
1,2023-08-02,0.295964
2,2023-08-03,0.261900
3,2023-08-04,0.132408
4,2023-08-05,0.035578
